In [75]:
import pandas as pd
import numpy as np
# import torch
from torch import tensor, from_numpy, nn, optim, float32
from torchvision import transforms
from sklearn.preprocessing import Normalizer, LabelBinarizer
from sklearn.model_selection import train_test_split

In [76]:
df = pd.read_csv("housing.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity,median_house_value
0,-117.61,34.13,21.0,8416.0,1386.0,4308.0,1341.0,4.4611,INLAND,164600.0
1,-117.37,33.98,52.0,201.0,44.0,130.0,24.0,2.0250,INLAND,125000.0
2,-118.34,33.89,36.0,2274.0,411.0,1232.0,423.0,5.3730,<1H OCEAN,244500.0
3,-118.92,35.13,29.0,1297.0,262.0,909.0,253.0,1.9236,INLAND,106300.0
4,-121.80,37.23,18.0,3179.0,526.0,1663.0,507.0,5.9225,<1H OCEAN,265800.0


In [77]:
df.isnull().any()

longitude             False
latitude              False
housing_median_age    False
total_rooms           False
total_bedrooms         True
population            False
households            False
median_income         False
ocean_proximity       False
median_house_value    False
dtype: bool

In [78]:
df_filled = df.fillna(method="bfill", axis=1)
df_filled

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity,median_house_value
0,-117.61,34.13,21.0,8416.0,1386.0,4308.0,1341.0,4.4611,INLAND,164600.0
1,-117.37,33.98,52.0,201.0,44.0,130.0,24.0,2.025,INLAND,125000.0
2,-118.34,33.89,36.0,2274.0,411.0,1232.0,423.0,5.373,<1H OCEAN,244500.0
3,-118.92,35.13,29.0,1297.0,262.0,909.0,253.0,1.9236,INLAND,106300.0
4,-121.8,37.23,18.0,3179.0,526.0,1663.0,507.0,5.9225,<1H OCEAN,265800.0
...,...,...,...,...,...,...,...,...,...,...
16507,-119.53,36.55,34.0,2065.0,343.0,1041.0,313.0,3.2917,INLAND,111500.0
16508,-122.4,37.73,50.0,1947.0,411.0,1170.0,384.0,3.4769,NEAR BAY,238700.0
16509,-118.41,33.92,29.0,1436.0,401.0,674.0,343.0,3.6389,<1H OCEAN,275000.0
16510,-117.08,32.62,36.0,1674.0,309.0,818.0,307.0,3.4773,NEAR OCEAN,150400.0


In [79]:
binarizer = LabelBinarizer()
df_filled["ocean_proximity"] = binarizer.fit_transform(df_filled["ocean_proximity"])
df_filled

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity,median_house_value
0,-117.61,34.13,21.0,8416.0,1386.0,4308.0,1341.0,4.4611,0,164600.0
1,-117.37,33.98,52.0,201.0,44.0,130.0,24.0,2.025,0,125000.0
2,-118.34,33.89,36.0,2274.0,411.0,1232.0,423.0,5.373,1,244500.0
3,-118.92,35.13,29.0,1297.0,262.0,909.0,253.0,1.9236,0,106300.0
4,-121.8,37.23,18.0,3179.0,526.0,1663.0,507.0,5.9225,1,265800.0
...,...,...,...,...,...,...,...,...,...,...
16507,-119.53,36.55,34.0,2065.0,343.0,1041.0,313.0,3.2917,0,111500.0
16508,-122.4,37.73,50.0,1947.0,411.0,1170.0,384.0,3.4769,0,238700.0
16509,-118.41,33.92,29.0,1436.0,401.0,674.0,343.0,3.6389,1,275000.0
16510,-117.08,32.62,36.0,1674.0,309.0,818.0,307.0,3.4773,0,150400.0


In [80]:
x = df_filled.drop("median_house_value", axis=1)
y = df_filled["median_house_value"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1)
x_train

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
3961,-122.13,37.75,36.0,768.0,93.0,229.0,93.0,5.3602,0
2552,-118.27,33.94,34.0,721.0,165.0,661.0,171.0,2.0789,1
7749,-122.72,38.58,4.0,7042.0,1100.0,2936.0,1043.0,5.0555,1
11311,-117.31,34.08,40.0,2011.0,495.0,1528.0,469.0,1.9375,0
13153,-117.98,33.64,20.0,1851.0,495.0,792.0,363.0,3.8187,0
...,...,...,...,...,...,...,...,...,...
8495,-117.25,33.21,13.0,1203.0,292.0,1035.0,293.0,2.6339,1
16117,-119.6,36.66,27.0,1388.0,296.0,1056.0,284.0,1.6094,0
15525,-118.24,34.08,52.0,109.0,20.0,86.0,24.0,4.9844,1
6497,-118.09,33.89,42.0,1150.0,215.0,708.0,204.0,3.6875,1


In [81]:
normalizer = Normalizer()
x_train_norm = normalizer.fit_transform(x_train)
x_test_norm = normalizer.transform(x_test)

In [82]:
from_numpy(y_train.values.astype(np.float32))

tensor([330000.,  92400., 240800.,  ..., 187500., 171500., 500001.])

In [83]:
x_train_tensor = from_numpy(x_train_norm).to(float32)
x_test_tensor = from_numpy(x_test_norm).to(float32)
y_train_tensor = from_numpy(y_train.values.astype(np.float32))
y_test_tensor = from_numpy(y_test.values.astype(np.float32))

In [120]:
class NeuralNetwork(nn.Module):
	def __init__(self):
		super(NeuralNetwork, self).__init__()
		self.flatten = nn.Flatten()
		self.layer_stack = nn.Sequential(
			nn.Linear(9, 32),
			nn.ReLU(),
			nn.Linear(32, 32),
			nn.ReLU(),
			nn.Linear(32, 1),
			nn.Sigmoid(),
		)

	def forward(self, x):
		x = self.flatten(x)
		logits = self.layer_stack(x)
		return logits

model = NeuralNetwork().to('cpu')
print(model)
# list(model.named_parameters())
# model.layer_stack[0].weight.data

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (layer_stack): Sequential(
    (0): Linear(in_features=9, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
    (5): Sigmoid()
  )
)


In [126]:
len(y_train_tensor) == len(x_train_tensor)

True

In [121]:
criterion = nn.MSELoss()
optimiser = optim.SGD(model.parameters(), lr = 0.001)

In [ ]:
n_epochs = 100
optimiser.zero_grad()
for epoch in range(n_epochs):
	# optimiser.zero_grad()
	y_hat = model(x_train_tensor)
	loss = criterion(y_hat, y_train_tensor)
	loss.backward()
	optimiser.step()

	print(f"Epoch: {epoch}\t w: {model.layer_stack[0].weight.data[0]}\t b: {model.layer_stack[0].bias.data[0]:.4f} \t L: {loss:.4f}")
	